# Unit 5: 模型评估与可视化

## 学习目标
- 掌握模型性能评估方法
- 学会使用多种评估指标
- 掌握混淆矩阵的绘制与分析
- 学会使用TensorBoard进行训练可视化
- 理解过拟合与欠拟合的诊断方法
- 掌握模型预测结果的可视化技巧

## 参考资源
- [PyTorch官方文档 - TensorBoard](https://pytorch.org/docs/stable/tensorboard.html)
- [scikit-learn评估指标](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics)
- [PyTorch教程 - TensorBoard](https://pytorch.org/tutorials/intermediate/tensorboard_tutorial.html)

## 5.1 评估指标

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 60)
print("5.1 常用评估指标")
print("=" * 60)

y_true = np.array([0, 1, 2, 0, 1, 2, 1, 0, 2, 1])
y_pred = np.array([0, 1, 2, 0, 0, 2, 1, 1, 2, 1])

accuracy = accuracy_score(y_true, y_pred)
print(f"准确率(Accuracy): {accuracy:.4f}")
print(f"  定义: 正确预测的样本比例")

precision = precision_score(y_true, y_pred, average='macro')
print(f"\n精确率(Precision, macro): {precision:.4f}")
print(f"  定义: 预测为正类的样本中真正为正类的比例")

recall = recall_score(y_true, y_pred, average='macro')
print(f"\n召回率(Recall, macro): {recall:.4f}")
print(f"  定义: 真正为正类的样本中被预测为正类的比例")

f1 = f1_score(y_true, y_pred, average='macro')
print(f"\nF1分数(F1-Score, macro): {f1:.4f}")
print(f"  定义: 精确率和召回率的调和平均")

print(f"\n分类报告:")
print(classification_report(y_true, y_pred, target_names=['Class 0', 'Class 1', 'Class 2']))

## 5.2 混淆矩阵

In [ ]:
print("=" * 60)
print("5.2 混淆矩阵可视化")
print("=" * 60)

torch.manual_seed(42)
num_samples = 500
num_classes = 5

y_true_large = torch.randint(0, num_classes, (num_samples,)).numpy()
y_pred_large = y_true_large.copy()

noise_indices = np.random.choice(num_samples, size=int(num_samples * 0.2), replace=False)
y_pred_large[noise_indices] = np.random.randint(0, num_classes, size=len(noise_indices))

cm = confusion_matrix(y_true_large, y_pred_large)
print(f"混淆矩阵:\n{cm}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'Class {i}' for i in range(num_classes)],
            yticklabels=[f'Class {i}' for i in range(num_classes)],
            ax=ax)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14)
plt.tight_layout()
plt.show()

cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=[f'Class {i}' for i in range(num_classes)],
            yticklabels=[f'Class {i}' for i in range(num_classes)],
            ax=ax)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Normalized Confusion Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 5.3 使用PyTorch计算评估指标

In [ ]:
print("=" * 60)
print("5.3 PyTorch中计算评估指标")
print("=" * 60)

def calculate_accuracy(outputs, targets):
    _, predicted = outputs.max(1)
    correct = predicted.eq(targets).sum().item()
    total = targets.size(0)
    return 100.0 * correct / total

logits = torch.randn(100, 10)
targets = torch.randint(0, 10, (100,))

acc = calculate_accuracy(logits, targets)
print(f"批次准确率: {acc:.2f}%")

def top_k_accuracy(outputs, targets, k=5):
    _, predicted = outputs.topk(k, dim=1)
    correct = predicted.eq(targets.unsqueeze(1)).sum().item()
    total = targets.size(0)
    return 100.0 * correct / total

top1_acc = top_k_accuracy(logits, targets, k=1)
top5_acc = top_k_accuracy(logits, targets, k=5)
print(f"\nTop-1准确率: {top1_acc:.2f}%")
print(f"Top-5准确率: {top5_acc:.2f}%")

## 5.4 TensorBoard可视化

In [ ]:
print("=" * 60)
print("5.4 TensorBoard可视化")
print("=" * 60)

try:
    from torch.utils.tensorboard import SummaryWriter
    
    writer = SummaryWriter('runs/cnn_experiment')
    
    for epoch in range(50):
        train_loss = 2.0 * np.exp(-epoch / 20) + 0.1 * np.random.randn()
        val_loss = 2.5 * np.exp(-epoch / 25) + 0.15 * np.random.randn()
        train_acc = 100 * (1 - np.exp(-epoch / 15)) + 2 * np.random.randn()
        val_acc = 95 * (1 - np.exp(-epoch / 18)) + 3 * np.random.randn()
        
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('Accuracy/train', train_acc, epoch)
        writer.add_scalar('Accuracy/val', val_acc, epoch)
    
    dummy_input = torch.randn(1, 3, 32, 32)
    model = nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(16 * 16 * 16, 10)
    )
    writer.add_graph(model, dummy_input)
    
    writer.close()
    print("TensorBoard日志已保存至 runs/cnn_experiment")
    print("运行 'tensorboard --logdir=runs' 查看可视化结果")
    
except ImportError:
    print("TensorBoard未安装，跳过此示例")
    print("安装命令: pip install tensorboard")

## 5.5 过拟合与欠拟合诊断

In [ ]:
print("=" * 60)
print("5.5 过拟合与欠拟合诊断")
print("=" * 60)

epochs = np.arange(1, 51)

underfit_train = 2.0 - 0.01 * epochs + 0.05 * np.random.randn(50)
underfit_val = 2.2 - 0.008 * epochs + 0.06 * np.random.randn(50)

good_train = 2.0 * np.exp(-epochs / 15) + 0.1
good_val = 2.5 * np.exp(-epochs / 18) + 0.15

overfit_train = 2.0 * np.exp(-epochs / 10) + 0.05
overfit_val = 2.0 * np.exp(-epochs / 10) + 0.05 + 0.5 * (1 - np.exp(-epochs / 20))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs, underfit_train, 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs, underfit_val, 'r-', label='Val Loss', linewidth=2)
axes[0].set_title('Underfitting', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, good_train, 'b-', label='Train Loss', linewidth=2)
axes[1].plot(epochs, good_val, 'r-', label='Val Loss', linewidth=2)
axes[1].set_title('Good Fitting', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, overfit_train, 'b-', label='Train Loss', linewidth=2)
axes[2].plot(epochs, overfit_val, 'r-', label='Val Loss', linewidth=2)
axes[2].set_title('Overfitting', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n诊断指南:")
print("欠拟合: 训练损失和验证损失都很高，且差距小")
print("  解决方法: 增加模型复杂度、训练更多epoch、减少正则化")
print("\n良好拟合: 训练损失和验证损失都较低，差距小")
print("  这是理想的训练状态")
print("\n过拟合: 训练损失很低，但验证损失较高且逐渐上升")
print("  解决方法: 增加正则化、Dropout、数据增强、早停法")

## 5.6 早停法(Early Stopping)实现

In [ ]:
print("=" * 60)
print("5.6 早停法实现")
print("=" * 60)

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
    
    def __call__(self, current_score):
        if self.best_score is None:
            self.best_score = current_score
            return False
        
        if self.mode == 'min':
            is_improved = current_score < self.best_score - self.min_delta
        else:
            is_improved = current_score > self.best_score + self.min_delta
        
        if is_improved:
            self.best_score = current_score
            self.counter = 0
        else:
            self.counter += 1
            print(f"  EarlyStopping计数器: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        
        return self.early_stop

early_stopping = EarlyStopping(patience=5, mode='min')

val_losses_sim = [2.0, 1.8, 1.6, 1.5, 1.4, 1.35, 1.36, 1.37, 1.38, 1.39, 1.40, 1.41]

print("模拟验证损失变化:")
for epoch, val_loss in enumerate(val_losses_sim, 1):
    print(f"\nEpoch {epoch}: Val Loss = {val_loss:.4f}")
    should_stop = early_stopping(val_loss)
    if should_stop:
        print(f"\n早停触发! 在第{epoch}个epoch停止训练")
        break

## 5.7 模型预测可视化

In [ ]:
print("=" * 60)
print("5.7 模型预测结果可视化")
print("=" * 60)

num_images = 8
num_classes = 10
class_names = [f'Class {i}' for i in range(num_classes)]

images = torch.randn(num_images, 3, 32, 32)
true_labels = torch.randint(0, num_classes, (num_images,))
predictions = torch.randint(0, num_classes, (num_images,))
probabilities = torch.softmax(torch.randn(num_images, num_classes), dim=1)

rows = 2
cols = 4
fig, axes = plt.subplots(rows, cols, figsize=(16, 8))
axes = axes.ravel()

for i in range(num_images):
    img = images[i].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min())
    
    axes[i].imshow(img)
    
    color = 'green' if predictions[i] == true_labels[i] else 'red'
    axes[i].set_title(f'True: {class_names[true_labels[i]]}\nPred: {class_names[predictions[i]]}',
                     color=color, fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i in range(num_images):
    probs = probabilities[i].numpy()
    axes[i].bar(range(num_classes), probs, color='steelblue')
    axes[i].axvline(x=true_labels[i], color='green', linestyle='--', linewidth=2, label='True')
    axes[i].axvline(x=predictions[i], color='red', linestyle='--', linewidth=2, label='Pred')
    axes[i].set_xlabel('Class')
    axes[i].set_ylabel('Probability')
    axes[i].set_title(f'Image {i+1} Probabilities')
    axes[i].set_xticks(range(num_classes))
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 本章小结

本单元我们学习了：
1. 常用评估指标(准确率、精确率、召回率、F1分数)
2. 混淆矩阵的计算与可视化
3. Top-K准确率计算
4. TensorBoard训练可视化
5. 过拟合与欠拟合的诊断方法
6. 早停法(Early Stopping)的实现
7. 模型预测结果的可视化

## 练习建议
1. 在实际训练中使用TensorBoard监控训练过程
2. 尝试不同的评估指标组合
3. 分析混淆矩阵找出模型容易混淆的类别
4. 实现早停法并应用到训练中

## 下一步
进入Unit 6，学习高级CNN架构与迁移学习技术。